In [ ]:
import pandas as pd
accelerometer = pd.read_csv('Accelerometer.csv')
gyroscope = pd.read_csv('Gyroscope.csv')
print(accelerometer.info())
print(gyroscope.info())


<class 'pandas.DataFrame'>
RangeIndex: 102051 entries, 0 to 102050
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Timestamp     102051 non-null  str    
 1   Milliseconds  102051 non-null  int64  
 2   X             102051 non-null  float64
 3   Y             102051 non-null  float64
 4   Z             102051 non-null  float64
dtypes: float64(3), int64(1), str(1)
memory usage: 3.9 MB
None
<class 'pandas.DataFrame'>
RangeIndex: 105541 entries, 0 to 105540
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Timestamp     105541 non-null  str    
 1   Milliseconds  105541 non-null  int64  
 2   X             105541 non-null  float64
 3   Y             105541 non-null  float64
 4   Z             105541 non-null  float64
dtypes: float64(3), int64(1), str(1)
memory usage: 4.0 MB
None


In [5]:
print(accelerometer.describe())
print(gyroscope.describe())

       Milliseconds              X              Y              Z
count  1.020510e+05  102051.000000  102051.000000  102051.000000
mean   5.479418e+05       0.185906      -1.596479       9.883985
std    3.006162e+05       0.434955       0.879152       0.575385
min    2.000000e+00     -15.995361      -7.577072       0.806580
25%    3.352480e+05       0.311554      -1.312042      10.095489
50%    5.445090e+05       0.311554      -1.312042      10.095489
75%    7.708460e+05       0.311554      -1.312042      10.095489
max    1.020502e+06      11.261444      12.452820      19.663315
       Milliseconds              X              Y              Z
count  1.055410e+05  105541.000000  105541.000000  105541.000000
mean   5.641430e+05      -0.001560       0.175945       0.059691
std    3.083199e+05       0.083217       0.168070       0.110592
min    1.000000e+00      -2.691254      -5.099960      -6.423645
25%    3.352530e+05       0.010269       0.035690       0.007889
50%    5.445270e+05      

In [10]:
# Remove duplicate rows from original dataframes
accelerometer = accelerometer.drop_duplicates()
gyroscope = gyroscope.drop_duplicates()

print("Accelerometer rows:", len(accelerometer))
print("Gyroscope rows:", len(gyroscope))

Accelerometer rows: 8445
Gyroscope rows: 11848


In [11]:
acc = accelerometer[["Milliseconds", "X", "Y", "Z"]].copy()
gyro = gyroscope[["Milliseconds", "X", "Y", "Z"]].copy()

acc.rename(columns={
    "X": "Acc_X",
    "Y": "Acc_Y",
    "Z": "Acc_Z"
}, inplace=True)

gyro.rename(columns={
    "X": "Gyro_X",
    "Y": "Gyro_Y",
    "Z": "Gyro_Z"
}, inplace=True)

print("Accelerometer rows:", len(acc))
print("Gyroscope rows:", len(gyro))

print("Accelerometer unique timestamps:", acc["Milliseconds"].nunique())
print("Gyroscope unique timestamps:", gyro["Milliseconds"].nunique())

Accelerometer rows: 8445
Gyroscope rows: 11848
Accelerometer unique timestamps: 8445
Gyroscope unique timestamps: 11848


Merging

In [ ]:
merged_data = pd.merge(acc,gyro, on="Milliseconds", how="inner")

print("Merged rows:", len(merged_data))
print(merged_data.head())

Merged rows: 1733
   Milliseconds     Acc_X     Acc_Y     Acc_Z    Gyro_X    Gyro_Y    Gyro_Z
0            22 -0.061935  7.509399  6.369507  0.062454  0.108658 -0.025665
1            52 -0.009262  7.560867  6.245621 -0.009979 -0.151260 -0.049637
2            92 -0.357620  7.621323  5.779343 -0.113312 -0.302002 -0.117813
3           192 -0.217560  7.478867  5.905045 -0.078690 -0.096939  0.019608
4           242 -0.140945  7.496231  6.028336 -0.004654  0.031952  0.068604


In [13]:
import numpy as np

merged_data["Acceleration_Magnitude"] = np.sqrt(merged_data["Acc_X"]**2 + merged_data["Acc_Y"]**2 + merged_data["Acc_Z"]**2)

merged_data["Gyroscope_Magnitude"] = np.sqrt(merged_data["Gyro_X"]**2 + merged_data["Gyro_Y"]**2 + merged_data["Gyro_Z"]**2)

print(merged_data.head())

print(merged_data[["Acceleration_Magnitude", "Gyroscope_Magnitude"]].describe())

   Milliseconds     Acc_X     Acc_Y     Acc_Z    Gyro_X    Gyro_Y    Gyro_Z  \
0            22 -0.061935  7.509399  6.369507  0.062454  0.108658 -0.025665   
1            52 -0.009262  7.560867  6.245621 -0.009979 -0.151260 -0.049637   
2            92 -0.357620  7.621323  5.779343 -0.113312 -0.302002 -0.117813   
3           192 -0.217560  7.478867  5.905045 -0.078690 -0.096939  0.019608   
4           242 -0.140945  7.496231  6.028336 -0.004654  0.031952  0.068604   

   Acceleration_Magnitude  Gyroscope_Magnitude  
0                9.847108             0.127929  
1                9.806864             0.159509  
2                9.571481             0.343402  
3                9.531544             0.126387  
4                9.620508             0.075822  
       Acceleration_Magnitude  Gyroscope_Magnitude
count             1733.000000          1733.000000
mean                10.115170             0.242865
std                  1.073985             0.515211
min                  4.5660

In [14]:
merged_data["Acceleration_Change"] = (merged_data["Acceleration_Magnitude"].diff().abs())
merged_data["Gyroscope_Change"] = (merged_data["Gyroscope_Magnitude"].diff().abs())
print(merged_data[["Acceleration_Magnitude", "Gyroscope_Magnitude", "Acceleration_Change", "Gyroscope_Change"]].describe())

       Acceleration_Magnitude  Gyroscope_Magnitude  Acceleration_Change  \
count             1733.000000          1733.000000          1732.000000   
mean                10.115170             0.242865             0.640712   
std                  1.073985             0.515211             1.077552   
min                  4.566092             0.002224             0.000000   
25%                  9.750344             0.062069             0.000000   
50%                 10.185157             0.127929             0.216122   
75%                 10.334526             0.299573             0.808548   
max                 17.848700             6.632796             9.063540   

       Gyroscope_Change  
count       1732.000000  
mean           0.055862  
std            0.228317  
min            0.000000  
25%            0.000000  
50%            0.010467  
75%            0.038673  
max            3.394745  


In [15]:
acc_threshold = (
    merged_data["Acceleration_Change"].mean()
    + 2 * merged_data["Acceleration_Change"].std()
)

gyro_threshold = (
    merged_data["Gyroscope_Magnitude"].mean()
    + 2 * merged_data["Gyroscope_Magnitude"].std()
)

print("Acceleration threshold:", acc_threshold)
print("Gyroscope threshold:", gyro_threshold)



Acceleration threshold: 2.7958166861869818
Gyroscope threshold: 1.273287406360153


In [16]:
# Classification

merged_data["Event"] = "Normal Movement"

# Turning: high gyroscope magnitude
merged_data.loc[
    merged_data["Gyroscope_Magnitude"] > gyro_threshold,
    "Event"
] = "Turning"

# Sudden acceleration/braking
# Applied only where rotation is not already classified as turning
merged_data.loc[
    (merged_data["Acceleration_Change"] > acc_threshold) &
    (merged_data["Gyroscope_Magnitude"] <= gyro_threshold),
    "Event"
] = "Sudden Acceleration/Braking"

# Display results
print(merged_data[
    ["Milliseconds",
     "Acceleration_Change",
     "Gyroscope_Magnitude",
     "Event"]
].head(20))

    Milliseconds  Acceleration_Change  Gyroscope_Magnitude            Event
0             22                  NaN             0.127929  Normal Movement
1             52             0.040244             0.159509  Normal Movement
2             92             0.235383             0.343402  Normal Movement
3            192             0.039937             0.126387  Normal Movement
4            242             0.088964             0.075822  Normal Movement
5            292             0.084159             0.221244  Normal Movement
6            322             0.041317             0.101592  Normal Movement
7            362             0.113282             0.069900  Normal Movement
8            402             0.247467             0.156967  Normal Movement
9            452             0.171852             0.069978  Normal Movement
10           482             0.058872             0.052305  Normal Movement
11           582             0.196736             0.080706  Normal Movement
12          

In [17]:
print("\nEvent Classification:")
print(merged_data["Event"].value_counts())


Event Classification:
Event
Normal Movement                1639
Sudden Acceleration/Braking      53
Turning                          41
Name: count, dtype: int64
